## 🎯 Learning Objectives
* Understand the core components of a LangGraph workflow: State, Nodes, and Edges.
* Learn how to define a graph's state using TypedDict and Annotated.
* Implement simple nodes as Python functions that interact with the graph state.
* Construct a basic LangGraph workflow using `StateGraph` and `MessageGraph`.
* Execute a LangGraph workflow and interpret its output.


## Your First LangGraph Workflow: Building a Stateful AI Agent

Welcome to the foundational lesson on LangGraph! In the rapidly evolving landscape of AI, building agents that can maintain context, make decisions, and interact with tools over multiple steps is crucial. LangGraph, an extension of LangChain, provides a powerful framework for orchestrating these complex, stateful interactions.

### What is LangGraph?

Imagine you're building a sophisticated AI assistant. It doesn't just answer a single question; it engages in a conversation, remembers previous turns, decides when to use a tool (like a calculator or a search engine), and then incorporates the tool's results back into the conversation. This isn't a linear process; it's a dynamic flow with branching paths and loops.

LangGraph helps you define these dynamic flows using a graph structure. Think of it like designing a flowchart or a state machine for your AI agent. Each step in the agent's reasoning or action sequence is a 'node', and the transitions between these steps are 'edges'. The 'state' of the graph keeps track of all the information the agent needs to remember as it moves through these steps.

### Core Concepts:

1.  **State**: This is the single source of truth for your agent. It's a Python dictionary or a `TypedDict` that holds all the relevant information, such as conversational messages, tool outputs, user preferences, or any other data the agent needs to operate. LangGraph ensures this state is passed between nodes and updated consistently.

2.  **Nodes**: These are the 'actors' or 'steps' in your workflow. A node is typically a Python function that takes the current graph state as input, performs some operation (e.g., calls an LLM, executes a tool, processes data), and then returns an update to the state. Nodes are the building blocks of your agent's logic.

3.  **Edges**: Edges define how your agent transitions between nodes. They can be:
    *   **Direct Edges**: Unconditionally move from one node to another.
    *   **Conditional Edges**: Based on the output of a node (often a 'router' node), the graph decides which of several possible next nodes to execute. This is where the agent's decision-making logic resides.

### The Power of LangGraph

By explicitly defining state, nodes, and edges, LangGraph offers several key advantages:

*   **Clarity**: Visualizing and understanding complex agent logic becomes much easier.
*   **Control**: You have granular control over every step of your agent's execution, making debugging and optimization straightforward.
*   **Robustness**: It naturally handles multi-turn interactions, tool usage, and error recovery by managing state transitions reliably.
*   **ReAct Patterns**: It's perfectly suited for implementing advanced reasoning patterns like ReAct (Reasoning and Acting), where an LLM reasons about what to do next, acts, and then observes the results.

### Building Our First Workflow: A Simple Conversational Agent

We'll construct a basic agent that can respond to user input. For demonstration purposes, our 'LLM' will be a simple Python function that simulates an LLM's response, and we'll introduce a conditional edge to show how the agent can decide its next step. This will lay the groundwork for more complex agents capable of tool use and advanced reasoning.


In [ ]:
import operator
from typing import Annotated, TypedDict

# LangGraph imports
from langgraph.graph import StateGraph, END

# LangChain imports (for messages, though we'll use simple strings for LLM output here)
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage

# 1. Define the Graph State
# This TypedDict defines the structure of our agent's state.
# 'messages' will accumulate all conversational turns.
# Annotated[list[BaseMessage], operator.add] means that when a node returns a list of messages,
# they should be appended to the existing 'messages' list in the state.
class AgentState(TypedDict):
    messages: Annotated[list[BaseMessage], operator.add]

# 2. Define the Nodes
# Each node is a function that takes the current state and returns an update to the state.

def call_llm(state: AgentState) -> dict:
    """Simulates an LLM call and returns a message to add to the state."""
    messages = state["messages"]
    last_message = messages[-1].content if messages else ""

    # Simple mock LLM logic
    if "tool" in last_message.lower() or "search" in last_message.lower():
        response_content = "Okay, I'll use a tool. What do you need to search for?"
        next_step = "tool_needed"
    elif "hello" in last_message.lower() or "hi" in last_message.lower():
        response_content = "Hello there! How can I assist you today?"
        next_step = "continue"
    else:
        response_content = f"You said: '{last_message}'. I'm a simple AI. Is there anything else?"
        next_step = "continue"

    print(f"--- LLM Node Executed ---")
    print(f"LLM Response: {response_content}")
    return {"messages": [AIMessage(content=response_content)], "next_step": next_step}

def call_tool(state: AgentState) -> dict:
    """Simulates a tool call and returns a message with the tool's result."""
    messages = state["messages"]
    last_message = messages[-1].content if messages else ""

    # Simple mock tool logic
    tool_result = f"Tool executed for: '{last_message}'. Result: 'Data from external source'."

    print(f"--- Tool Node Executed ---")
    print(f"Tool Result: {tool_result}")
    return {"messages": [AIMessage(content=tool_result)]}

# 3. Define the Conditional Edge (Router)
# This function determines the next node based on the output of the previous node.

def decide_next_step(state: AgentState) -> str:
    """Decides whether to call a tool or end the conversation."""
    # The 'next_step' key is added by the 'call_llm' node for routing purposes
    if "next_step" in state and state["next_step"] == "tool_needed":
        print("--- Router: Decided to call tool ---")
        return "call_tool"
    else:
        print("--- Router: Decided to continue/end ---")
        return "end"

# 4. Build the Graph
# Initialize a StateGraph with our defined state.
workflow = StateGraph(AgentState)

# Add nodes to the graph
workflow.add_node("llm", call_llm)
workflow.add_node("tool", call_tool)

# Set the entry point for the graph
workflow.set_entry_point("llm")

# Add edges
# From 'llm', conditionally move to 'tool' or 'end' based on 'decide_next_step'
workflow.add_conditional_edges(
    "llm",          # Source node
    decide_next_step, # Function to determine next node
    {
        "call_tool": "tool", # If decide_next_step returns "call_tool", go to "tool" node
        "end": END           # If decide_next_step returns "end", terminate the graph
    }
)

# From 'tool', we'll go back to the LLM to process the tool's result (a simple loop)
workflow.add_edge("tool", "llm")

# 5. Compile the Graph
# This finalizes the graph structure, making it ready to be invoked.
app = workflow.compile()

# 6. Run the Graph (Invoke the Agent)
print("\n--- Running Agent with 'Hello' ---")
inputs = {"messages": [HumanMessage(content="Hello, AI!")]}
for s in app.stream(inputs):
    print(s)

print("\n--- Running Agent with 'I need to use a tool' ---")
inputs = {"messages": [HumanMessage(content="I need to use a tool to find some data.")]}
for s in app.stream(inputs):
    print(s)

print("\n--- Running Agent with 'What is the weather?' ---")
inputs = {"messages": [HumanMessage(content="What is the weather like today?")]}
for s in app.stream(inputs):
    print(s)


### Interpreting the Output and Understanding the Flow

The output from the code cell demonstrates the execution of our first LangGraph workflow. Let's break down what you saw:

1.  **`--- LLM Node Executed ---`**: This indicates that the `call_llm` function was invoked. It received the initial `HumanMessage` from our `inputs`.
2.  **`LLM Response: ...`**: This is the simulated response from our `call_llm` node, which then gets added to the `messages` list in the graph's state.
3.  **`--- Router: Decided to ... ---`**: The `decide_next_step` function was called. It inspected the `next_step` key (which `call_llm` added to the state) to determine the next transition.
    *   **For "Hello, AI!"**: The `call_llm` node returned `"next_step": "continue"`. The router then returned `"end"`, leading to the `END` node, and the graph terminated.
    *   **For "I need to use a tool..."**: The `call_llm` node detected the keyword "tool" and returned `"next_step": "tool_needed"`. The router then returned `"call_tool"`, directing the graph to the `tool` node.
    *   **For "What is the weather...?"**: Similar to "Hello, AI!", the `call_llm` node returned `"next_step": "continue"`, leading to `END`.
4.  **`--- Tool Node Executed ---`**: When the router directed the flow to the `tool` node, the `call_tool` function was invoked. It simulated a tool's action and added its result as an `AIMessage` to the state.
5.  **Looping back to LLM**: Notice that after the `tool` node, the graph printed `--- LLM Node Executed ---` again. This is due to our `workflow.add_edge("tool", "llm")` line. This creates a loop, allowing the LLM to process the tool's output and potentially decide on further actions or to finally end the conversation. This is a fundamental pattern for ReAct agents.

Each dictionary printed by `app.stream(inputs)` represents the state of the graph after each node's execution. You can see how the `messages` list grows with each turn, demonstrating the stateful nature of LangGraph.

### Performance Trade-offs and Use Cases

**Performance Trade-offs:**

*   **Overhead**: LangGraph introduces a small amount of overhead compared to a simple, linear chain of LLM calls. This is due to the graph traversal logic, state management, and serialization/deserialization if using persistent state.
*   **Complexity Management**: For simple, single-turn interactions, LangGraph might be overkill. However, for anything beyond a basic prompt-response, the benefits of explicit state management and control far outweigh this minimal overhead.

**Typical Use Cases:**

*   **Complex Conversational Agents**: Building chatbots that can handle multi-turn dialogues, remember user preferences, and adapt their responses.
*   **Tool-Using Agents (ReAct)**: Agents that can decide when to use external tools (APIs, databases, search engines), execute them, and integrate the results into their reasoning.
*   **Autonomous Agents**: Agents that can plan, execute, and self-correct over extended periods, often involving multiple steps and decision points.
*   **Multi-Agent Systems**: Orchestrating interactions between several specialized AI agents, each handling a different part of a complex task.
*   **Human-in-the-Loop Workflows**: Designing systems where human intervention is required at specific points in an automated process.

This simple example provides a glimpse into the power of LangGraph. As you progress, you'll learn to build much more sophisticated agents by combining these foundational concepts with advanced features like persistent state, error handling, and more complex routing.


### Resources

*   **LangGraph Documentation**: The official and most comprehensive resource for learning LangGraph.
    *   [https://langchain-ai.github.io/langgraph/](https://langchain-ai.github.io/langgraph/)
*   **LangChain Documentation**: LangGraph builds upon LangChain, so understanding LangChain's core components (like messages, LLMs, tools) is beneficial.
    *   [https://python.langchain.com/docs/get_started/](https://python.langchain.com/docs/get_started/)
*   **LangGraph Tutorials**: Explore more examples and advanced patterns.
    *   [https://langchain-ai.github.io/langgraph/tutorials/](https://langchain-ai.github.io/langgraph/tutorials/)
*   **LangGraph GitHub Repository**: For diving into the source code or contributing.
    *   [https://github.com/langchain-ai/langgraph](https://github.com/langchain-ai/langgraph)
